# Improved Live AQI Model (Six Pollutants)

This notebook builds a **separate experimental model** using the six pollutant measurements commonly available from air-quality data services. It reads the existing `dataset/city_day.csv` and predicts its `AQI` column.

It does not change `app.py`, `models/aqi_model.pkl`, or the original EDA notebook. Running the final cell writes a new artifact, `models/aqi_model_live.pkl`. The website will still use the old model until its input form and route are deliberately updated later.


## What this experiment can and cannot show

We will compare Linear Regression, Random Forest, and Gradient Boosting with the same six numeric inputs: PM2.5, PM10, NO2, CO, SO2, and O3. City is included as a categorical feature.

The CSV contains daily observations from 2015 through 2020. This notebook can compare models on a later-date holdout from that dataset, but that does **not** prove live accuracy today or accuracy in an unseen village/city. No result here promises to match Google or a monitoring station.


## 1. Load the data and libraries

The path check makes this notebook work whether it is opened from the project root or from the `notebooks` folder. We parse dates now so later rows can be split chronologically.


In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'dataset' / 'city_day.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'dataset' / 'city_day.csv'
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

print('Dataset path:', DATA_PATH)
print('Rows and columns:', df.shape)
print('Date range:', df['Date'].min(), 'to', df['Date'].max())
print('Cities:', df['City'].nunique())


Dataset path: C:\Users\balar\OneDrive\Documents\AQI_PREDICTION\dataset\city_day.csv
Rows and columns: (29531, 16)
Date range: 2015-01-01 00:00:00 to 2020-07-01 00:00:00
Cities: 26


## 2. Check the selected columns and missingness

Rows without AQI cannot teach a supervised regression model what the target should be, so they will be excluded. Missing pollutant inputs are different: an sklearn pipeline will learn replacement medians from only the training portion. This keeps test-period information out of preprocessing.


In [2]:
POLLUTANT_FEATURES = ['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3']
FEATURES = ['City'] + POLLUTANT_FEATURES
TARGET = 'AQI'

required_columns = ['Date'] + FEATURES + [TARGET]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

print('Rows with missing AQI:', int(df[TARGET].isna().sum()))
print('Missing selected features:')
display(df[FEATURES].isna().sum().to_frame('missing rows'))
print('Percent missing in selected features:')
display(df[FEATURES].isna().mean().mul(100).round(1).to_frame('missing percent'))


Rows with missing AQI: 4681
Missing selected features:


,missing rows
City,0
PM2.5,4598
PM10,11140
NO2,3585
CO,2059
SO2,3854
O3,4022


Percent missing in selected features:


,missing percent
City,0.0
PM2.5,15.6
PM10,37.7
NO2,12.1
CO,7.0
SO2,13.1
O3,13.6


## 3. Prepare rows and split by date

We sort observations by date and reserve the latest 20% of distinct dates as a final test period. All cities on a date stay on the same side of the split. This better resembles predicting later observations than a random split. It still tests later dates for cities present in training; it is not an unseen-city test.


In [3]:
model_data = df[['Date'] + FEATURES + [TARGET]].copy()
model_data = model_data.dropna(subset=['Date', TARGET])
model_data = model_data.sort_values(['Date', 'City']).reset_index(drop=True)

all_dates = np.array(sorted(model_data['Date'].unique()))
split_index = int(len(all_dates) * 0.80)
split_date = all_dates[split_index]

train_frame = model_data[model_data['Date'] < split_date].copy()
test_frame = model_data[model_data['Date'] >= split_date].copy()

X_train = train_frame[FEATURES].copy()
y_train = train_frame[TARGET].copy()
X_test = test_frame[FEATURES].copy()
y_test = test_frame[TARGET].copy()

print('Split date:', split_date)
print('Training rows:', len(train_frame), '| dates:', train_frame['Date'].min(), 'to', train_frame['Date'].max())
print('Test rows:', len(test_frame), '| dates:', test_frame['Date'].min(), 'to', test_frame['Date'].max())

training_cities = set(train_frame['City'].dropna().unique())
test_cities = set(test_frame['City'].dropna().unique())
unseen_test_cities = sorted(test_cities - training_cities)
print('Cities appearing in test:', len(test_cities))
print('Test cities absent from training:', unseen_test_cities if unseen_test_cities else 'None')


Split date: 2019-05-27 00:00:00
Training rows: 16018 | dates: 2015-01-01 00:00:00 to 2019-05-26 00:00:00
Test rows: 8832 | dates: 2019-05-27 00:00:00 to 2020-07-01 00:00:00
Cities appearing in test: 26
Test cities absent from training: ['Aizawl', 'Bhopal', 'Chandigarh', 'Coimbatore', 'Ernakulam', 'Kochi', 'Shillong']


## 4. Build preprocessing and model pipelines

The numeric imputer replaces a missing pollutant with its median learned from that fold's training data. City is filled with the most common training city and one-hot encoded. `handle_unknown='ignore'` lets the pipeline run for a city not seen during fitting, but it does not teach the model that city's pollution pattern; an unseen city's output must be treated cautiously.

Each regressor is wrapped in the same preprocessing pipeline, so fitting and evaluation use consistent steps and do not fit imputers on test values.


In [4]:
numeric_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])
city_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_preprocessor, POLLUTANT_FEATURES),
    ('city', city_preprocessor, ['City'])
])

models = {
    'Linear Regression': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]),
    'Random Forest': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1))
    ]),
    'Gradient Boosting': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
    ])
}

print('Models ready:', ', '.join(models))


Models ready: Linear Regression, Random Forest, Gradient Boosting


## 5. Choose a model using time-ordered validation

We use three expanding-window validation folds inside the training period. Each fold trains on earlier dates and validates on later dates. We select the model with the lowest average validation RMSE, then use the untouched latest-date test period once to report MAE, RMSE, and R? for all three candidates. This keeps the final test period out of model selection.


In [5]:
def regression_metrics(actual, predicted):
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted),
    }

train_dates = np.array(sorted(train_frame['Date'].unique()))
date_cv = TimeSeriesSplit(n_splits=3)
validation_scores = {name: [] for name in models}

for fold_number, (fit_date_indices, valid_date_indices) in enumerate(date_cv.split(train_dates), start=1):
    fit_dates = train_dates[fit_date_indices]
    valid_dates = train_dates[valid_date_indices]
    fit_mask = train_frame['Date'].isin(fit_dates)
    valid_mask = train_frame['Date'].isin(valid_dates)

    X_fit = train_frame.loc[fit_mask, FEATURES]
    y_fit = train_frame.loc[fit_mask, TARGET]
    X_valid = train_frame.loc[valid_mask, FEATURES]
    y_valid = train_frame.loc[valid_mask, TARGET]

    print(f'Fold {fold_number}: fit through {pd.Timestamp(fit_dates[-1]).date()}, validate {pd.Timestamp(valid_dates[0]).date()} to {pd.Timestamp(valid_dates[-1]).date()}')
    for name, pipeline in models.items():
        fold_model = clone(pipeline)
        fold_model.fit(X_fit, y_fit)
        fold_prediction = fold_model.predict(X_valid)
        validation_scores[name].append(regression_metrics(y_valid, fold_prediction)['RMSE'])

cv_summary = pd.DataFrame({
    'Model': list(models.keys()),
    'Mean validation RMSE': [np.mean(validation_scores[name]) for name in models],
    'Validation RMSE std': [np.std(validation_scores[name]) for name in models],
}).sort_values('Mean validation RMSE').reset_index(drop=True)
display(cv_summary)
best_model_name = cv_summary.loc[0, 'Model']
print('Selected using training-period validation:', best_model_name)


Fold 1: fit through 2016-02-08, validate 2016-02-09 to 2017-03-15


Fold 2: fit through 2017-03-15, validate 2017-03-16 to 2018-04-20


Fold 3: fit through 2018-04-20, validate 2018-04-21 to 2019-05-26


,Model,Mean validation RMSE,Validation RMSE std
0,Random Forest,63.516880,7.305173
1,Gradient Boosting,63.689972,8.748093
2,Linear Regression,67.758723,5.163717


Selected using training-period validation: Random Forest


## 6. Evaluate all three models on the latest-date test period

These scores describe this one chronological holdout only. Lower MAE/RMSE and higher R? are generally preferable. Errors can still be substantial, and a good score on these 26 cities does not establish good performance for an unseen village.


In [6]:
test_models = {}
test_rows = []
for name, pipeline in models.items():
    fitted_model = clone(pipeline)
    fitted_model.fit(X_train, y_train)
    test_prediction = fitted_model.predict(X_test)
    test_models[name] = fitted_model
    row = {'Model': name, **regression_metrics(y_test, test_prediction)}
    row['Mean validation RMSE'] = np.mean(validation_scores[name])
    test_rows.append(row)

results = pd.DataFrame(test_rows).sort_values('Mean validation RMSE').reset_index(drop=True)
display(results.round(3))
print('Best by training-period validation:', best_model_name)


,Model,MAE,RMSE,R2,Mean validation RMSE
0,Random Forest,17.695,32.341,0.903,63.517
1,Gradient Boosting,18.461,31.803,0.906,63.690
2,Linear Regression,24.276,36.691,0.875,67.759


Best by training-period validation: Random Forest


## 7. Compare with the existing baseline, if its results file is available

The existing comparison CSV records metrics for the original 13-feature Gradient Boosting model. We show its baseline row next to the new experiment when present. Treat this as context, not a perfectly controlled head-to-head result: the existing artifact has more features, and its saved metrics may have come from a different training run or split. The new model is not automatically better just because it uses fewer inputs.


In [7]:
BASELINE_PATH = PROJECT_ROOT / 'models' / 'model_comparison.csv'
if BASELINE_PATH.exists():
    baseline_table = pd.read_csv(BASELINE_PATH)
    baseline_row = baseline_table[baseline_table['Model'].eq('Gradient Boosting')]
    if not baseline_row.empty:
        baseline_view = baseline_row.rename(columns={'R2 Score': 'R2'}).copy()
        baseline_view['Evaluation'] = 'Existing 13-feature baseline (saved results)'
        new_view = results[results['Model'].eq(best_model_name)][['MAE', 'RMSE', 'R2']].copy()
        new_view['Evaluation'] = f'New six-pollutant model: {best_model_name} (this notebook test period)'
        display(pd.concat([
            baseline_view[['Evaluation', 'MAE', 'RMSE', 'R2']],
            new_view[['Evaluation', 'MAE', 'RMSE', 'R2']]
        ], ignore_index=True).round(3))
    else:
        print('No original Gradient Boosting row found in the baseline comparison file.')
else:
    print('Baseline comparison file not found; skipping this optional comparison.')


,Evaluation,MAE,RMSE,R2
0,Existing 13-feature baseline (saved results),18.269,30.993,0.911
1,New six-pollutant model: Random Forest (this n...,17.695,32.341,0.903


## 8. Refit the selected pipeline and save it separately

The model choice comes from validation within the earlier training period. For the separate live artifact, we now fit that selected pipeline on all labeled rows, so it can use the full historical dataset. The held-out test scores above remain the evaluation results; they are not new scores for this refitted artifact.

This artifact expects **City plus six pollutant columns**. It is not compatible with the current website's 13-feature input until the website is intentionally updated. The original `aqi_model.pkl` is not overwritten.


In [8]:
final_pipeline = clone(models[best_model_name])
final_pipeline.fit(model_data[FEATURES], model_data[TARGET])

LIVE_MODEL_PATH = PROJECT_ROOT / 'models' / 'aqi_model_live.pkl'
joblib.dump(final_pipeline, LIVE_MODEL_PATH)
print('Saved separate six-pollutant pipeline to:', LIVE_MODEL_PATH)
print('Input feature names:', list(final_pipeline.feature_names_in_))


Saved separate six-pollutant pipeline to: C:\Users\balar\OneDrive\Documents\AQI_PREDICTION\models\aqi_model_live.pkl
Input feature names: ['City', 'PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3']


## 9. Limitations and what to do next

- `city_day.csv` ends in 2020, so this model is still based on historical conditions rather than current observations.
- City is one-hot encoded. An unseen city is accepted by the encoder but gets no learned city-specific effect. Village-level predictions are especially uncertain.
- The pipeline handles missing inputs with medians; it does not turn imputed values into measurements. Check live input coverage before trusting a prediction.
- For a stronger live evaluation, collect newer station-level observations for intended locations, align units and averaging periods, and hold out entire dates and cities/stations during evaluation.
- Compare estimates with CPCB or Google only at matching locations and times, and report differences. Do not alter predictions to force a match.

These results can establish whether the six-feature model is promising on the available holdout. They cannot guarantee greater accuracy or agreement with another provider.
